# 04 - Method C: Gradient-Boosted Regime Classifier

**Inertia v2 - Factor Regimes** - Sprint 2

Reframe regime detection as classification: label each training month *favorable*
if next-month factor return exceeds the in-sample median, then fit a gradient-boosted
tree classifier on lagged factor returns and rolling factor volatilities.
Output: $P(\text{favorable})$ per OOS month per factor.

Hyperparameters fixed a priori: 200 trees, depth 3, learning rate 0.05, row subsample 0.8.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.data import build_factor_panel, FF5_FACTORS
from lib.methods import fit_predict_gbm_oos
from lib.backtest import prob_to_weight, apply_weights, static_factor_returns
from lib.evaluation import perf_stats
from lib.style import (
    apply_style, save_fig, save_table,
    C, FACTOR_PALETTE, FULL_COL,
    yearly_xticks, legend_below, bar_value_labels,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
FIG_DIR, TABLE_DIR = '../figures', '../tables'
OOS_START = '1990-01-01'

In [2]:
panel = build_factor_panel(include_macro=False)
factor_features = [f'lag1_{f}' for f in FF5_FACTORS] + [f'vol6_{f}' for f in FF5_FACTORS]

probs = {}
for f in FF5_FACTORS:
    print(f'Fitting GBM for {f}...')
    probs[f] = fit_predict_gbm_oos(panel, f, factor_features,
                                     oos_start=OOS_START, refit_months=12, min_train=120)
P = pd.DataFrame(probs)
print(f'\nOOS shape: {P.shape}, range {P.index.min().date()} -> {P.index.max().date()}')
P.describe().T[['mean','std','min','max']]

Fitting GBM for MKT_RF...


Fitting GBM for SMB...


Fitting GBM for HML...


Fitting GBM for RMW...


Fitting GBM for CMA...



OOS shape: (433, 5), range 1990-01-31 -> 2026-01-31


,mean,std,min,max
MKT_RF,0.5085,0.1916,0.0548,0.9508
SMB,0.5130,0.1809,0.0604,0.9056
HML,0.5071,0.1920,0.0700,0.9274
RMW,0.4943,0.1751,0.0857,0.9395
CMA,0.4970,0.1813,0.0848,0.9397


In [3]:
fig, axes = plt.subplots(5, 1, figsize=(FULL_COL, 7.0), sharex=True)
for ax, f in zip(axes, FF5_FACTORS):
    s = P[f]
    ax.plot(s.index, s.values, color=FACTOR_PALETTE[f], linewidth=0.9)
    ax.axhline(0.5, color=C['muted'], linewidth=0.4, linestyle='--')
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel(f, color=FACTOR_PALETTE[f], fontsize=10, fontweight='bold')
    ax.set_yticks([0, 0.5, 1])
axes[0].set_title('Method C: GBM P(favorable), 1990 to 2026',
                  loc='left', color=C['ink'])
yearly_xticks(axes[-1], every=5)
save_fig(fig, '11_method_c_probs_timeseries', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/11_method_c_probs_timeseries.png


In [4]:
rows = {}
timed_returns = {}
for f in FF5_FACTORS:
    p = P[f].dropna()
    w = prob_to_weight(p, mode='linear')
    timed = apply_weights(w, panel[f], cost_bps_oneway=5.0)
    static = static_factor_returns(panel[f], timed.index)
    timed_returns[f] = timed['r_net']
    s_static = perf_stats(static)
    s_timed  = perf_stats(timed['r_net'])
    rows[f] = {
        'sharpe_static': s_static['sharpe_ann'],
        'sharpe_timed':  s_timed['sharpe_ann'],
        'mean_static':   s_static['mean_ann'],
        'mean_timed':    s_timed['mean_ann'],
        'vol_static':    s_static['vol_ann'],
        'vol_timed':     s_timed['vol_ann'],
        'mdd_static':    s_static['max_drawdown'],
        'mdd_timed':     s_timed['max_drawdown'],
    }
compare = pd.DataFrame(rows).T
compare['sharpe_uplift'] = compare['sharpe_timed'] - compare['sharpe_static']
save_table(compare, '11_method_c_per_factor_compare', out_dir=TABLE_DIR)
compare

  saved: ../tables/11_method_c_per_factor_compare.{csv,md}


,sharpe_static,sharpe_timed,mean_static,mean_timed,vol_static,vol_timed,mdd_static,mdd_timed,sharpe_uplift
MKT_RF,0.5993,0.1147,0.0906,0.0072,0.1512,0.0629,-0.5411,-0.2624,-0.4846
SMB,0.0908,-0.0552,0.0096,-0.0020,0.1059,0.0369,-0.4210,-0.3618,-0.1460
HML,0.1661,0.1022,0.0186,0.0045,0.1121,0.0438,-0.5779,-0.2246,-0.0640
RMW,0.4646,-0.0301,0.0418,-0.0010,0.0900,0.0341,-0.4178,-0.2466,-0.4947
CMA,0.2705,0.3654,0.0204,0.0112,0.0755,0.0306,-0.2725,-0.1007,0.0949


In [5]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.6))
x = np.arange(len(FF5_FACTORS))
w = 0.36
bars_s = ax.bar(x - w/2, compare['sharpe_static'].values, w, color=C['blue'],
                label='Static factor', edgecolor='white', linewidth=0.5)
bars_t = ax.bar(x + w/2, compare['sharpe_timed'].values, w, color=C['purple'],
                label='Method C timed', edgecolor='white', linewidth=0.5)
ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(FF5_FACTORS, fontsize=10)
ax.set_ylabel('Annualized Sharpe ratio')
ax.set_title('Method C: timed vs static Sharpe, by factor (1990 to 2026)',
             loc='left', color=C['ink'])
ax.set_ylim(min(compare[['sharpe_static','sharpe_timed']].min().min(), 0) - 0.10,
            compare[['sharpe_static','sharpe_timed']].max().max() + 0.20)
bar_value_labels(ax, bars_s, fmt='{:+.2f}', offset=0.03, fontsize=8, color=C['blue'])
bar_value_labels(ax, bars_t, fmt='{:+.2f}', offset=0.03, fontsize=8, color=C['purple'])
legend_below(ax, ncol=2)
save_fig(fig, '12_method_c_sharpe_bars', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/12_method_c_sharpe_bars.png


In [6]:
tr_df = pd.DataFrame(timed_returns).dropna(how='all')
method_c_composite = tr_df.mean(axis=1)
static_df = pd.DataFrame({f: panel[f].shift(-1).reindex(tr_df.index) for f in FF5_FACTORS}).dropna(how='all')
static_composite = static_df.mean(axis=1)
stats = pd.DataFrame({
    'static_eq_weight': perf_stats(static_composite),
    'method_c_timed':   perf_stats(method_c_composite),
}).T
save_table(stats, '12_method_c_composite_perf', out_dir=TABLE_DIR)
stats

  saved: ../tables/12_method_c_composite_perf.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
static_eq_weight,433.0000,0.0362,0.0491,0.7375,-0.1871,2.6831,-0.1508
method_c_timed,433.0000,0.0040,0.0208,0.1902,0.2490,2.9203,-0.0817


In [7]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.4))
for r, color, label, lw in [
    (static_composite, C['blue'],   'Equal-weight static FF5', 1.0),
    (method_c_composite, C['purple'], 'Method C composite (timed)', 1.4),
]:
    cum = (1 + r).cumprod()
    ax.plot(cum.index, cum.values, color=color, linewidth=lw, label=label)
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of \\$1 (log)')
ax.set_title('Method C composite vs static FF5, 1990 to 2026',
             loc='left', color=C['ink'])
yearly_xticks(ax, every=5)
legend_below(ax, ncol=2)
save_fig(fig, '13_method_c_composite_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/13_method_c_composite_cumret.png


In [8]:
P.to_csv(f'{TABLE_DIR}/13_method_c_probs.csv')
tr_df.to_csv(f'{TABLE_DIR}/14_method_c_timed_returns.csv')
print(f'Saved Method C outputs.')

Saved Method C outputs.
